# TrainLM: native TorchTPU trainer smoke

Open in Colab on a single-device Cloud TPU v5e runtime. One code cell clones a fresh source snapshot; the next runs a three-step `distilbert/distilgpt2` training and export smoke from a pinned Hugging Face commit. The optional resume mode compares an additional four-step reference run. This is experimental evidence, not performance or support certification.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-tpu-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "main",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")

## Run the trainer

The default runs eager training for three steps. Before downloading the model, a read-only package preflight checks the installed TorchTPU stack. Missing or conflicting training dependencies stop the run. The known Colab Torch-range mismatch and unrelated IPython/Jedi conflict are labeled experimental, so the opt-in smoke can still gather evidence. Set `RUN_COMPILE = True` only after eager succeeds. Set `RUN_RESUME = True` to compare resume with an uninterrupted four-step run. Resume requires eager mode. The model and tokenizer use a fixed Hub commit, recorded in the report. The child has a timeout and process-group cleanup; raw logs stay local. A bounded JSON report is saved for sharing.

In [ ]:
import json
import runpy

RUN_COMPILE = False
RUN_RESUME = False
if RUN_COMPILE and RUN_RESUME:
    raise ValueError("Run compiled and resume checks separately.")
session_dir = Path(tempfile.mkdtemp(prefix="trainlm-native-run-"))
report_path = session_dir / "trainlm_native_trainer_report.json"
inspect_profile = runpy.run_path(str(repo_root / "examples/torchtpu_t1_install_preflight.py"))["inspect_profile"]
assess_profile = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_trainer_preflight.py"))["assess_trainer_profile"]
preflight = assess_profile(inspect_profile(repo_root))
print(json.dumps(preflight, indent=2))
if not preflight["proceed"]:
    report = {"schema_version": 1, "scope": "single_device_native_trainer_smoke",
              "status": "blocked", "source_revision": source_revision,
              "preflight": preflight, "checks": {"train": "not_run",
              "export": "not_run", "compile": "not_run",
              "resume": "not_run", "exact_weight_match": "not_run"}}
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(f"shareable_report={report_path}")
    raise RuntimeError("Training stack preflight blocked the smoke; download the report for details.")
TIMEOUT_SECONDS = 1800 if (RUN_COMPILE or RUN_RESUME) else 900
run_process = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_probe_process.py"))["run_probe_process"]
command = [sys.executable, str(repo_root / "examples/torchtpu_native_trainer_smoke.py"),
           "--output-dir", str(session_dir / "artifacts")]
if RUN_COMPILE:
    command.append("--compile")
if RUN_RESUME:
    command.append("--resume")
result = run_process(command, cwd=repo_root, timeout=TIMEOUT_SECONDS)
(session_dir / "stdout.log").write_text(result["stdout"])
(session_dir / "stderr.log").write_text(result["stderr"])
build_report = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_trainer_evidence.py"))["build_trainer_report"]
report = build_report(result, source_revision=source_revision,
                      compile_requested=RUN_COMPILE, resume_requested=RUN_RESUME)
report["preflight"] = preflight
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
print(json.dumps(report, indent=2))
print(f"shareable_report={report_path}")
print(f"local_artifacts_and_logs={session_dir}")
if report["status"] != "passed":
    raise RuntimeError("Native trainer smoke failed; inspect local logs before retrying.")

## Download evidence

Run this cell after the trainer cell, including when that cell reports a failed check. Share only the JSON report; raw logs may contain local details.

In [ ]:
from google.colab import files
files.download(str(report_path))

## If it fails

Keep the first error in the local logs and the source revision. A detected TPU must not be labeled a CPU pass. Torch 2.14.x remains outside TrainLM's declared package range; the owner-run CPU suite and native package profile must be reviewed before changing it.